# DIVE-HMM · CCBF — Notebook 2 (Headline experiments)

This notebook runs the **paper's headline evaluation** for Calibrated Condition-Belief Filtering (CCBF):

1. **Multi-condition localization** — CCBF vs single-image, fixed-observation filter, SeqSLAM-style, and an
   uncertainty baseline (SUE / L2), on genuinely multi-condition data.
2. **Calibration** — ECE and **reliability diagrams** (the headline reliability metric).
3. **Open-set / off-map** — **AUROC / AUPR** for "lost" detection.
4. **Theory validation** — the **ECE-vs-sequence-length (T)** curve that directly tests *Theorem 1
   (miscalibration propagation)*: a fixed, condition-agnostic observation model should degrade with T while CCBF
   stays flat.
5. **Label-free test-time domain adaptation** — condition belief vs. true condition, and condition-drift robustness.

**Datasets.** The robust backbone is **Nordland Full** (4 synchronized seasons, clean frame-ID GT → ideal,
*controlled* multi-condition testbed for calibration/open-set; note these metrics are **not** saturated the way
recall is). **RobotCar Seasons v2** and **Extended CMU** are added as hard real-world conditions via a *best-effort,
self-probing* loader — if the pose-GT format matches the standard schema it runs; otherwise it prints the format so
the loader can be finalized in NB3.

> Run top-to-bottom. `CONFIG.SMOKE_TEST=True` (default) uses a small subset so the full suite finishes in minutes.
> Offline / Kaggle Internet OFF. Reuses the descriptor cache in `/kaggle/working/dive_hmm/features` if present.

In [ ]:
# === P0.0 — Imports, device, seeds (same contract as NB1) ==============================
import os, sys, json, time, glob, math, warnings
from dataclasses import dataclass, field, asdict
from collections import OrderedDict, defaultdict
import numpy as np

REPORT = OrderedDict()
def log(step, status, msg=""):
    REPORT[step] = {"status": status, "msg": msg}
    print({"ok": "[ OK ]", "warn": "[WARN]", "fail": "[FAIL]", "info": "[info]"}.get(status, "[??]"), step + ":", msg)

def _seed(s=0):
    import random; random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

try:
    import torch; _HAS_TORCH = True
except Exception as e:
    _HAS_TORCH = False; print("torch not importable:", e)
_seed(0)
if _HAS_TORCH:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    BF16_OK = (DEVICE == "cuda" and torch.cuda.is_bf16_supported())
    print("torch", torch.__version__, "| device", DEVICE, "| bf16", BF16_OK)
    if DEVICE == "cuda": print("GPU:", torch.cuda.get_device_properties(0).name)
else:
    DEVICE, BF16_OK = "cpu", False
print("numpy", np.__version__)

In [ ]:
# === P0.1 — CONFIG =====================================================================
@dataclass
class Config:
    SMOKE_TEST: bool = True
    SEED: int = 0
    INPUT_ROOTS: tuple = ("/kaggle/input",)
    WORKING_DIR: str = "/kaggle/working/dive_hmm"
    CACHE_SUBDIR: str = "features"; CKPT_SUBDIR: str = "ckpt"

    IMG_SIZE: int = 322
    BATCH_SIZE: int = 64
    NUM_WORKERS: int = 2
    USE_BF16: bool = True
    SMOKE_MAX_FRAMES: int = 800          # per season/condition in smoke

    # CCBF
    STICKY_RHO: float = 0.97
    MOTION_STD: float = 2.0
    MOTION_VELOCITY: float = 1.0
    MOTION_LEAK: float = 1e-3
    NULL_PRIOR: float = 0.02
    NULL_SELF: float = 0.9

    # evaluation
    RECALL_KS: tuple = (1, 5, 10)
    PLACE_TOL: int = 2
    ECE_BINS: int = 15
    USE_TRAIN_CONDITION_LABELS: bool = True

    # experiments
    T_GRID: tuple = (1, 2, 3, 5, 8, 13, 21, 34, 55, 89)   # sequence lengths for ECE-vs-T
    OFFMAP_MAP_FRACTION: float = 0.7      # first 70% of the route is the map; last 30% are off-map queries
    ROBOTCAR_REF_COND: str = "overcast-reference"
    ROBOTCAR_QUERY_CONDS: tuple = ("night", "night-rain", "snow", "rain", "dusk", "sun")
    POSE_TOL_M: float = 5.0               # metres: a retrieval is correct if within this of GT pose

    @property
    def cache_dir(self): return os.path.join(self.WORKING_DIR, self.CACHE_SUBDIR)
    @property
    def ckpt_dir(self): return os.path.join(self.WORKING_DIR, self.CKPT_SUBDIR)

CONFIG = Config()
if _HAS_TORCH and DEVICE == "cuda":
    g = torch.cuda.get_device_properties(0).total_memory / 1e9
    CONFIG.BATCH_SIZE = 256 if g >= 70 else (128 if g >= 40 else 64)
CONFIG.USE_BF16 = CONFIG.USE_BF16 and BF16_OK
os.makedirs(CONFIG.cache_dir, exist_ok=True); os.makedirs(CONFIG.ckpt_dir, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(CONFIG).items()}, indent=0)[:800])
log("P0.1.config", "ok", f"SMOKE={CONFIG.SMOKE_TEST} batch={CONFIG.BATCH_SIZE} bf16={CONFIG.USE_BF16}")

In [ ]:
# === P0.2 — Offline env + path discovery ==============================================
for k, v in {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
             "PIP_NO_INDEX": "1", "WANDB_MODE": "offline", "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ.setdefault(k, v)

def _find_dir(markers_any, markers_all=(), roots=None, max_depth=4):
    roots = roots or CONFIG.INPUT_ROOTS; seen = set()
    for root in roots:
        if not os.path.isdir(root): continue
        frontier = [(root, 0)]
        while frontier:
            d, dep = frontier.pop(0)
            if d in seen or dep > max_depth: continue
            seen.add(d)
            try: entries = set(os.listdir(d))
            except Exception: continue
            ok_all = all(any(m.lower() in e.lower() for e in entries) for m in markers_all) if markers_all else True
            ok_any = any(any(m.lower() in e.lower() for e in entries) for m in markers_any) if markers_any else True
            if ok_all and ok_any: return d
            for e in sorted(entries):
                sub = os.path.join(d, e)
                if os.path.isdir(sub): frontier.append((sub, dep + 1))
    return None

PATHS = {
    "bundle":        _find_dir(["checkpoints"], ["checkpoints", "sources"]),
    "nordland_full": _find_dir(["data"], ["data", "annotations"]),
    "nordland_short":_find_dir(["summer", "winter"], ["summer", "winter"]),
    "robotcar":      _find_dir(["images"], ["images", "3D-models"]),
    "cmu":           _find_dir(["slice2", "slice6", "slice7"]),
}
for n, p in PATHS.items():
    log(f"P0.2.path.{n}", "ok" if p else "warn", p or "not found")
if PATHS["bundle"]:
    for sub in ["salad", "dinov2", "Hierarchical-Localization", "LightGlue", ""]:
        p = os.path.join(PATHS["bundle"], "sources", sub) if sub else os.path.join(PATHS["bundle"], "sources")
        if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p)

In [ ]:
# === P0.3 — Pluggable feature extractor (SALAD -> DINOv2 CLS+mean -> placeholder) ======
# NOTE: DINOv2 tokens are SIGNED; we use CLS + mean pooling (the GeM-clamp bug from NB1 v1 is fixed here).
EXTRACTOR_INFO = {"name": None, "dim": None, "detail": ""}

def _try_salad():
    """serizba/salad via helper.get_* (released dino_salad.ckpt configs), bypassing VPRModel. dim=8448."""
    if not PATHS["bundle"]: return None
    ck = os.path.join(PATHS["bundle"], "checkpoints", "dino_salad.ckpt")
    salad_dir = os.path.join(PATHS["bundle"], "sources", "salad")
    if not (os.path.isfile(ck) and os.path.isdir(salad_dir)): return None
    try:
        if salad_dir not in sys.path: sys.path.insert(0, salad_dir)
        from models import helper
        backbone = helper.get_backbone("dinov2_vitb14",
            {"num_trainable_blocks": 4, "return_token": True, "norm_layer": True})
        aggregator = helper.get_aggregator("SALAD",
            {"num_channels": 768, "num_clusters": 64, "cluster_dim": 128, "token_dim": 256})
        sd = torch.load(ck, map_location="cpu"); sd = sd.get("state_dict", sd)
        bb = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
        ag = {k[len("aggregator."):]: v for k, v in sd.items() if k.startswith("aggregator.")}
        backbone.load_state_dict(bb, strict=False); aggregator.load_state_dict(ag, strict=False)
        model = torch.nn.Module(); model.backbone = backbone; model.aggregator = aggregator
        model.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x): return model.aggregator(model.backbone(x))
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all() or d.ndim != 2 or d.shape[1] < 1000: return None
        return fwd, d.shape[1]
    except Exception:
        import traceback; print("  SALAD error:"); traceback.print_exc(); return None

def _try_dinov2():
    pth = os.path.join(PATHS["bundle"], "checkpoints", "dinov2_vitb14_pretrain.pth") if PATHS["bundle"] else None
    try:
        vit_base = __import__("dinov2.models.vision_transformer", fromlist=["vit_base"]).vit_base
        vit = vit_base(patch_size=14, img_size=518, init_values=1.0, block_chunks=0)
    except Exception as e:
        print("  DINOv2 build failed:", repr(e)[:120]); return None
    try:
        if pth and os.path.isfile(pth):
            sd = torch.load(pth, map_location="cpu"); sd = sd.get("state_dict", sd)
            m, u = vit.load_state_dict(sd, strict=False); print(f"  DINOv2 weights (missing={len(m)})")
        vit.eval().to(DEVICE)
        @torch.no_grad()
        def fwd(x):
            f = vit.forward_features(x)
            if isinstance(f, dict):
                parts = []
                if f.get("x_norm_clstoken") is not None: parts.append(f["x_norm_clstoken"])
                if f.get("x_norm_patchtokens") is not None: parts.append(f["x_norm_patchtokens"].mean(1))
                return torch.cat(parts, 1) if parts else f
            return f.mean(1) if f.dim() == 3 else f
        with torch.no_grad():
            d = fwd(torch.zeros(2, 3, CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, device=DEVICE))
        if not torch.isfinite(d).all(): return None
        return fwd, d.shape[1]
    except Exception as e:
        print("  DINOv2 error:", repr(e)[:120]); return None

def _placeholder():
    GRID = 4
    @torch.no_grad()
    def fwd(x):
        B, C, H, W = x.shape; gh, gw = H // GRID, W // GRID; feats = []
        for i in range(GRID):
            for j in range(GRID):
                t = x[:, :, i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                feats += [t.mean((2, 3)), t.std((2, 3)), (t[:, :, :, 1:] - t[:, :, :, :-1]).abs().mean((2, 3))]
        return torch.cat(feats, 1)
    return fwd, GRID * GRID * 9

def build_extractor():
    for name, b in [("salad", _try_salad), ("dinov2", _try_dinov2)]:
        if PATHS["bundle"]:
            print("try extractor:", name); r = b()
            if r is not None:
                EXTRACTOR_INFO.update(name=name, dim=r[1], detail="bundle")
                log("P0.3.extractor", "ok", f"{name} dim={r[1]}"); return r[0]
    fwd, dim = _placeholder(); EXTRACTOR_INFO.update(name="placeholder", dim=dim, detail="DEGRADED")
    log("P0.3.extractor", "warn", f"placeholder dim={dim} (not publication-grade)"); return fwd

EXTRACT_FWD = build_extractor()

## Shared library — metrics, baselines, and the CCBF core

The place/condition filter, calibrated observation model, and metrics are the **validated core from NB1**
(same equations as `CCBF_methodology.md`). We add three evaluation tools used by the headline experiments:
a **reliability diagram**, **AUROC** for open-set detection, and the **SUE / L2** uncertainty baseline
(Zaffar et al., CVPR 2024: the L2 distance to the nearest reference is a strong training-free uncertainty).

In [ ]:
# === L.1 — Metrics =====================================================================
def cosine_sim(Q, R):
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    R = R / (np.linalg.norm(R, axis=1, keepdims=True) + 1e-12)
    return Q @ R.T

def _logsumexp(x, axis=None):
    x = np.asarray(x, dtype=np.float64)
    m = np.max(x, axis=axis, keepdims=True)
    s = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return float(s.reshape(())) if axis is None else np.squeeze(s, axis=axis)

def recall_at_k(sim, q_ids, r_ids, ks=(1, 5, 10), tol=2):
    r_ids = np.asarray(r_ids); q_ids = np.asarray(q_ids)
    kmax = min(max(ks), sim.shape[1]); topk = np.argsort(-sim, axis=1)[:, :kmax]; out = {}
    for k in ks:
        kk = min(k, sim.shape[1])
        out[k] = float((np.abs(r_ids[topk[:, :kk]] - q_ids[:, None]) <= tol).any(1).mean())
    return out

def ece_from_conf(conf, correct, n_bins=15):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    bins = np.linspace(0, 1, n_bins + 1); ece, rows, N = 0.0, [], len(conf)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0: continue
        c, a, w = conf[m].mean(), correct[m].mean(), m.sum() / N
        ece += w * abs(a - c); rows.append((0.5 * (lo + hi), c, a, int(m.sum())))
    return float(ece), rows

def pred_correct(pred_idx, q_ids, r_ids, tol=2):
    return (np.abs(np.asarray(r_ids)[pred_idx] - np.asarray(q_ids)) <= tol).astype(float)

def fit_temperature(sim, gt_idx, grid=None):
    grid = grid if grid is not None else np.logspace(-2.2, 0.3, 30)
    v = gt_idx >= 0; s, g = sim[v], gt_idx[v]
    best, bn = grid[0], np.inf
    for tau in grid:
        lp = s / tau; lp = lp - _logsumexp(lp, axis=1)[:, None]
        nll = -lp[np.arange(len(g)), g].mean()
        if nll < bn: bn, best = nll, tau
    return float(best), float(bn)

def auroc(scores, labels):
    """Mann-Whitney AUROC; higher score => predicted positive. labels in {0,1}."""
    scores = np.asarray(scores, float); labels = np.asarray(labels, int)
    P, N = int(labels.sum()), int((labels == 0).sum())
    if P == 0 or N == 0: return float("nan")
    ranks = np.argsort(np.argsort(scores)) + 1               # ascending ranks, ties approx
    return float((ranks[labels == 1].sum() - P * (P + 1) / 2) / (P * N))

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores)); y = np.asarray(labels, int)[order]
    tp = np.cumsum(y); prec = tp / (np.arange(len(y)) + 1); rec = tp / max(1, y.sum())
    drec = np.diff(np.concatenate([[0], rec])); return float((prec * drec).sum())

def sue_uncertainty(sim):
    """SUE/L2 (Zaffar'24): L2 distance to nearest reference = sqrt(2 - 2*max cos). Higher => less certain."""
    return np.sqrt(np.maximum(2 - 2 * sim.max(1), 0.0))

In [ ]:
# === L.2 — Motion model + fixed-observation Bayes filter (baseline) ====================
def make_motion_kernel(velocity, std):
    lo, hi = int(math.floor(velocity - 4 * std)), int(math.ceil(velocity + 4 * std))
    offs = np.arange(lo, hi + 1); w = np.exp(-0.5 * ((offs - velocity) / max(std, 1e-3)) ** 2)
    return offs, w / w.sum()

def predict_place(place, offs, w, leak):
    M = place.shape[0]; new = np.zeros_like(place)
    for off, wt in zip(offs, w):
        if off == 0: new += wt * place
        elif off > 0:
            if off < M: new[off:] += wt * place[:M - off]
        else:
            a = -off
            if a < M: new[:M - a] += wt * place[a:]
    s = new.sum()
    if s > 0: new /= s
    return (1 - leak) * new + leak / M

def run_place_filter(sim, tau, null_quantile=0.98, exit_prob=0.01):
    T, M = sim.shape; offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    for t in range(T):
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        logb = np.empty(M + 1)
        logb[:M] = np.log(pp + 1e-30) + sim[t] / tau
        logb[M] = np.log(pn + 1e-30) + np.quantile(sim[t], null_quantile) / tau
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
    return preds, confs, nulls

In [ ]:
# === L.3 — Condition model + CCBF (validated core from NB1) ============================
def _pairwise_sq_dists(X, C):
    return np.maximum((X * X).sum(1)[:, None] + (C * C).sum(1)[None, :] - 2.0 * (X @ C.T), 0.0)

def cond_log_resp(desc, COND):
    logr = -0.5 * _pairwise_sq_dists(desc.astype(np.float32), COND["C"]) / (COND["sigma"] ** 2)
    return logr - _logsumexp(logr, axis=1)[:, None]

def _sticky_predict(bc, rho, K):
    return rho * bc + (1.0 - rho) * (1.0 - bc) / max(K - 1, 1)

def fit_condition_model(DESC, TRACKS):
    names = list(DESC.keys())
    if CONFIG.USE_TRAIN_CONDITION_LABELS and len(names) >= 2:
        C = np.stack([DESC[s].mean(0) for s in names]).astype(np.float32)
    else:
        pool = np.concatenate([DESC[s] for s in names], 0); K = max(2, len(names))
        rng = np.random.default_rng(CONFIG.SEED); C = pool[rng.choice(len(pool), K, replace=False)].copy()
        for _ in range(25):
            a = np.argmin(_pairwise_sq_dists(pool, C), 1)
            nC = np.stack([pool[a == k].mean(0) if (a == k).any() else C[k] for k in range(K)])
            if np.allclose(nC, C, atol=1e-5): C = nC; break
            C = nC
        names = [f"cond{k}" for k in range(K)]
    pool = np.concatenate([DESC[s] for s in DESC], 0)
    sigma = float(np.sqrt(np.median(_pairwise_sq_dists(pool, C).min(1))) + 1e-6)
    tau, season_list = [], list(DESC.keys())
    for i, s in enumerate(names):
        src = season_list[i] if s in DESC else season_list[i % len(season_list)]
        ref = ([o for o in season_list if o != src] or [src])[0]
        sims = cosine_sim(DESC[src], DESC[ref])
        gt = gt_match_index(TRACKS[src], TRACKS[ref]) if (src in TRACKS and ref in TRACKS) else np.arange(sims.shape[0])
        tr = (np.arange(sims.shape[0]) % 2 == 0) & (gt >= 0)
        if tr.sum() < 5: tr = (gt >= 0)
        tau.append(fit_temperature(sims[tr], gt[tr])[0])
    return dict(names=names, C=C, sigma=sigma, tau=np.asarray(tau, np.float32))

def run_ccbf(sim, query_desc, COND, null_quantile=0.98, exit_prob=0.01, return_traj=False):
    T, M = sim.shape; K = len(COND["tau"])
    inv_tau = (1.0 / np.maximum(COND["tau"], 1e-4)).astype(np.float64)
    appear = cond_log_resp(query_desc, COND); offs, w = make_motion_kernel(CONFIG.MOTION_VELOCITY, CONFIG.MOTION_STD)
    place = np.full(M, (1 - CONFIG.NULL_PRIOR) / M); null = CONFIG.NULL_PRIOR; bc = np.full(K, 1.0 / K)
    preds = np.empty(T, np.int64); confs = np.empty(T); nulls = np.empty(T)
    traj = np.empty((T, K)) if return_traj else None
    for t in range(T):
        bc = _sticky_predict(bc, CONFIG.STICKY_RHO, K)
        lbc = np.log(bc + 1e-30) + appear[t]; lbc -= _logsumexp(lbc); bc = np.exp(lbc)
        pp = (1 - exit_prob) * predict_place(place, offs, w, CONFIG.MOTION_LEAK) + null * (1 - CONFIG.NULL_SELF) / M
        pn = CONFIG.NULL_SELF * null + exit_prob * place.sum()
        A = sim[t][None, :] * inv_tau[:, None] + np.log(bc + 1e-30)[:, None]
        ll_place = _logsumexp(A, axis=0)
        ll_null = _logsumexp(np.log(bc + 1e-30) + np.quantile(sim[t], null_quantile) * inv_tau)
        logb = np.empty(M + 1); logb[:M] = np.log(pp + 1e-30) + ll_place; logb[M] = np.log(pn + 1e-30) + ll_null
        logb -= _logsumexp(logb); b = np.exp(logb)
        place, null = b[:M], b[M]; s = place.sum() + null; place, null = place / s, null / s
        preds[t], confs[t], nulls[t] = int(np.argmax(place)), float(place.max()), float(null)
        if return_traj: traj[t] = bc
    return (preds, confs, nulls, traj) if return_traj else (preds, confs, nulls)

def gt_match_index(q, r):
    pos = {fid: j for j, fid in enumerate(r.frame_ids)}
    return np.array([pos.get(fid, -1) for fid in q.frame_ids], dtype=np.int64)

log("lib", "ok", "metrics + baselines + CCBF core loaded")

In [ ]:
# === L.4 — Reliability diagram helper ==================================================
def plot_reliability(curves, title, fname=None):
    """curves: dict name -> (conf, correct). Draws reliability diagram + ECE in legend."""
    try:
        import matplotlib.pyplot as plt
    except Exception as e:
        print("(plot skipped:", repr(e)[:60], ")"); return
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    for name, (conf, corr) in curves.items():
        ece, rows = ece_from_conf(conf, corr, CONFIG.ECE_BINS)
        if rows:
            xs = [r[1] for r in rows]; ys = [r[2] for r in rows]
            ax.plot(xs, ys, "o-", ms=4, label=f"{name} (ECE={ece:.3f})")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title(title)
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    if fname:
        out = os.path.join(CONFIG.WORKING_DIR, fname); fig.tight_layout(); fig.savefig(out, dpi=110); print("saved", out)
    plt.show()

## Data — Nordland (up to 4 synchronized seasons)

Seasons are row-aligned on the common frame-id grid, so row *i* is the same place under every season (clean GT).
With Nordland Full this gives **four genuine conditions** (spring/summer/fall/winter); with only Short it falls
back to summer/winter. Descriptors are cached (reused across NB1/NB2).

In [ ]:
# === D.1 — Track, transform, cached extraction =========================================
from PIL import Image
def _frame_id(f):
    d = "".join(c for c in os.path.splitext(f)[0] if c.isdigit()); return int(d) if d else None
def _imgs(d): return [f for f in os.listdir(d) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if d and os.path.isdir(d) else []

class Track:
    def __init__(self, name, condition, image_dir):
        self.name, self.condition, self.image_dir = name, condition, image_dir
        pairs = sorted([(_frame_id(f), f) for f in _imgs(image_dir) if _frame_id(f) is not None])
        self.frame_ids = [a for a, _ in pairs]; self.paths = [os.path.join(image_dir, b) for _, b in pairs]
    def __len__(self): return len(self.paths)

try:
    import torchvision.transforms as T
    def build_transform(sz): return T.Compose([T.Resize((sz, sz), antialias=True), T.ToTensor(),
                                                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    _HAS_TV = True
except Exception:
    _HAS_TV = False
    def build_transform(sz):
        mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)
        def tx(im):
            a = (np.asarray(im.resize((sz, sz)).convert("RGB"), np.float32) / 255.0 - mean) / std
            return torch.from_numpy(a.transpose(2, 0, 1).copy())
        return tx
if not _HAS_TV: CONFIG.NUM_WORKERS = 0

class ImageList(torch.utils.data.Dataset):
    def __init__(self, paths, sz): self.paths = paths; self.tx = build_transform(sz); self.sz = sz
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: im = Image.open(self.paths[i]).convert("RGB")
        except Exception: im = Image.new("RGB", (self.sz, self.sz))
        return self.tx(im), i

@torch.no_grad()
def extract_paths(paths, frame_ids, tag, force=False):
    cp = os.path.join(CONFIG.cache_dir, tag + ".npy"); mp = os.path.join(CONFIG.cache_dir, tag + ".json")
    if (not force) and os.path.isfile(cp) and os.path.isfile(mp):
        man = json.load(open(mp))
        if man.get("extractor") == EXTRACTOR_INFO["name"] and man.get("frame_ids") == list(frame_ids):
            print("  cache hit:", tag); return np.load(cp).astype(np.float32)
    dl = torch.utils.data.DataLoader(ImageList(paths, CONFIG.IMG_SIZE), batch_size=CONFIG.BATCH_SIZE,
                                     shuffle=False, num_workers=CONFIG.NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    feats = np.empty((len(paths), EXTRACTOR_INFO["dim"]), np.float32)
    amp = torch.bfloat16 if CONFIG.USE_BF16 else torch.float16; t0 = time.time()
    for imgs, idx in dl:
        imgs = imgs.to(DEVICE, non_blocking=True)
        if DEVICE == "cuda":
            with torch.autocast("cuda", dtype=amp): d = EXTRACT_FWD(imgs)
        else: d = EXTRACT_FWD(imgs)
        feats[idx.numpy()] = torch.nn.functional.normalize(d.float(), dim=1).cpu().numpy()
    np.save(cp, feats); json.dump({"extractor": EXTRACTOR_INFO["name"], "dim": int(EXTRACTOR_INFO["dim"]),
                                   "frame_ids": list(frame_ids)}, open(mp, "w"))
    assert np.isfinite(feats).all(), f"non-finite descriptors in {tag}"
    print(f"  extracted {tag}: {feats.shape} in {time.time()-t0:.1f}s"); return feats

In [ ]:
# === D.2 — Build aligned Nordland season tracks + descriptors ==========================
def _season_dir(root, season):
    cands = [c for c, _, fs in os.walk(root) if os.path.basename(c).lower() == season
             and any(f.lower().endswith((".png", ".jpg")) for f in fs)]
    return sorted(cands, key=lambda p: -p.count(os.sep))[0] if cands else None

# Prefer the 4-season FULL set (subsampled in smoke) so multi-condition experiments have real conditions;
# fall back to the 2-season Short set only if Full is not attached.
NORD_ROOT = PATHS["nordland_full"] or PATHS["nordland_short"]
ALL_SEASONS = ["spring", "summer", "fall", "winter"]
raw = {}
if NORD_ROOT:
    for s in ALL_SEASONS:
        d = _season_dir(NORD_ROOT, s)
        if d: raw[s] = Track(f"nordland_{s}", s, d)
# align on common frame-ids; subsample in smoke
TRACKS = {}
if len(raw) >= 2:
    common = sorted(set.intersection(*[set(t.frame_ids) for t in raw.values()]))
    if CONFIG.SMOKE_TEST and len(common) > CONFIG.SMOKE_MAX_FRAMES:
        sel = np.linspace(0, len(common) - 1, CONFIG.SMOKE_MAX_FRAMES).round().astype(int)
        common = [common[i] for i in sorted(set(sel.tolist()))]
    cset = set(common)
    for s, t in raw.items():
        keep = [(f, p) for f, p in zip(t.frame_ids, t.paths) if f in cset]
        t.frame_ids = [f for f, _ in keep]; t.paths = [p for _, p in keep]; TRACKS[s] = t
    log("D.2.tracks", "ok", f"seasons={list(TRACKS)} | aligned frames={len(common)}")
else:
    log("D.2.tracks", "fail", "need >=2 Nordland seasons")

DESC = {}
for s, t in TRACKS.items():
    DESC[s] = extract_paths(t.paths, t.frame_ids, tag=f"nordland_{'smoke' if CONFIG.SMOKE_TEST else 'full'}_{s}")
log("D.2.desc", "ok" if DESC else "fail", f"descriptors: {{{', '.join(f'{k}:{v.shape}' for k,v in DESC.items())}}}" if DESC else "none")

## Experiment A — Nordland headline suite (robust, clean GT)

Reference map = one season; queries = the other season(s). Because tracks are row-aligned, GT place = identity.
- **A1** multi-condition comparison (single-image / fixed filter / CCBF) — recall + ECE.
- **A2** reliability diagrams.
- **A3** **ECE vs sequence length** — the direct test of *Theorem 1*: a single-τ filter should grow miscalibrated as
  it accumulates frames; CCBF (per-condition τ) should stay flat.
- **A4** **open-set / off-map AUROC** — map = first 70% of the route; queries entering the last 30% are off-map and
  must be flagged (CCBF null posterior vs fixed-filter null vs SUE/L2).
- **A5** **label-free TTA** — condition-belief accuracy and condition-drift tracking.

In [ ]:
# === A.0 — Condition model + episode helpers ==========================================
RESULTS2 = OrderedDict()
RUN_A = len(DESC) >= 2
if RUN_A:
    COND = fit_condition_model(DESC, TRACKS)
    SEASONS = list(DESC.keys())
    REF = "summer" if "summer" in SEASONS else SEASONS[0]
    QUERIES = [s for s in SEASONS if s != REF]
    print(f"ref map = {REF} | query conditions = {QUERIES}")
    print("per-condition tau:", {n: round(float(t), 4) for n, t in zip(COND["names"], COND["tau"])})
    log("A.0", "ok", f"ref={REF}, queries={QUERIES}, K={len(COND['names'])}")
else:
    log("A.0", "fail", "need >=2 seasons for Experiment A")

def episode(ref, qs):
    sim = cosine_sim(DESC[qs], DESC[ref])
    gt = gt_match_index(TRACKS[qs], TRACKS[ref])
    return sim, gt, np.asarray(TRACKS[qs].frame_ids), np.asarray(TRACKS[ref].frame_ids)

def fit_global_tau(ref, queries):
    S, G = [], []
    for qs in queries:
        sim, gt, _, _ = episode(ref, qs); tr = (np.arange(len(gt)) % 2 == 0) & (gt >= 0)
        S.append(sim[tr]); G.append(gt[tr])
    return fit_temperature(np.concatenate(S), np.concatenate(G))[0]

In [ ]:
# === A.1 / A.2 — Multi-condition comparison + reliability =============================
def exp_A1(ref, queries, COND):
    tau_g = fit_global_tau(ref, queries)
    pools = {m: {"conf": [], "corr": []} for m in ["single_image", "fixed_filter", "ccbf"]}
    recall_rows = {m: [] for m in pools}
    for qs in queries:
        sim, gt, qid, rid = episode(ref, qs); ev = (np.arange(len(gt)) % 2 == 1) & (gt >= 0)
        # single-image
        lp = sim / tau_g; pr = np.exp(lp - _logsumexp(lp, axis=1)[:, None])
        corr = pred_correct(pr.argmax(1), qid, rid, CONFIG.PLACE_TOL)
        pools["single_image"]["conf"].append(pr.max(1)[ev]); pools["single_image"]["corr"].append(corr[ev])
        recall_rows["single_image"].append(recall_at_k(sim[ev], qid[ev], rid, CONFIG.RECALL_KS, CONFIG.PLACE_TOL))
        # fixed filter
        p, c, n = run_place_filter(sim, tau_g); corr = pred_correct(p, qid, rid, CONFIG.PLACE_TOL)
        pools["fixed_filter"]["conf"].append(c[ev]); pools["fixed_filter"]["corr"].append(corr[ev])
        recall_rows["fixed_filter"].append({1: float(corr[ev].mean())})
        # ccbf
        p, c, n = run_ccbf(sim, DESC[qs], COND); corr = pred_correct(p, qid, rid, CONFIG.PLACE_TOL)
        pools["ccbf"]["conf"].append(c[ev]); pools["ccbf"]["corr"].append(corr[ev])
        recall_rows["ccbf"].append({1: float(corr[ev].mean())})
    out = {}
    curves = {}
    for m in pools:
        conf = np.concatenate(pools[m]["conf"]); corr = np.concatenate(pools[m]["corr"])
        ece, _ = ece_from_conf(conf, corr, CONFIG.ECE_BINS)
        r1 = float(np.mean([r[1] for r in recall_rows[m]]))
        out[m] = {"R@1": r1, "ECE": ece, "acc": float(corr.mean())}
        curves[m] = (conf, corr)
    return out, curves

if RUN_A:
    a1, curves = exp_A1(REF, QUERIES, COND); RESULTS2["A1_comparison"] = a1
    print(f"{'method':<16}{'R@1':>8}{'ECE':>8}{'acc':>8}")
    for m, v in a1.items(): print(f"{m:<16}{v['R@1']:>8.3f}{v['ECE']:>8.3f}{v['acc']:>8.3f}")
    plot_reliability(curves, f"Reliability — Nordland (ref={REF}, {len(QUERIES)} query conds)", "A2_reliability.png")
    log("A.1", "ok", f"CCBF ECE={a1['ccbf']['ECE']:.3f} vs fixed ECE={a1['fixed_filter']['ECE']:.3f}")

In [ ]:
# === A.3 — ECE vs sequence length (Theorem 1 test) ====================================
def exp_A3_ece_vs_T(ref, queries, COND, n_windows=None):
    """Fresh-window design: for each sequence length T, run many length-T windows (filter reset each time)
    and compute ECE from the window's final-step (confidence, correctness). Theorem 1 predicts the fixed
    single-tau filter's ECE rises with T while CCBF stays flat. (Full-scale uses fewer windows; the per-step
    Python filter is vectorized to GPU in NB3.)"""
    n_windows = n_windows or (120 if CONFIG.SMOKE_TEST else 40)
    tau_g = fit_global_tau(ref, queries)
    curve = {"fixed_filter": [], "ccbf": []}
    for T in CONFIG.T_GRID:
        pools = {m: {"conf": [], "corr": []} for m in curve}
        for qs in queries:
            sim, gt, qid, rid = episode(ref, qs); L = len(gt)
            if L <= T: continue
            starts = sorted(set(np.linspace(0, L - T, min(n_windows, L - T + 1)).astype(int).tolist()))
            for st in starts:
                e = st + T - 1
                pf, cf, _ = run_place_filter(sim[st:st + T], tau_g)
                pc, cc, _ = run_ccbf(sim[st:st + T], DESC[qs][st:st + T], COND)
                pools["fixed_filter"]["conf"].append(cf[-1])
                pools["fixed_filter"]["corr"].append(float(abs(rid[pf[-1]] - qid[e]) <= CONFIG.PLACE_TOL))
                pools["ccbf"]["conf"].append(cc[-1])
                pools["ccbf"]["corr"].append(float(abs(rid[pc[-1]] - qid[e]) <= CONFIG.PLACE_TOL))
        for m in curve:
            conf = np.asarray(pools[m]["conf"]); corr = np.asarray(pools[m]["corr"])
            curve[m].append(ece_from_conf(conf, corr, max(5, CONFIG.ECE_BINS // 2))[0] if len(conf) > 10 else np.nan)
    print("T       :", CONFIG.T_GRID)
    for m in curve: print(f"ECE[{m:<13}]:", [round(x, 3) if np.isfinite(x) else None for x in curve[m]])
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 4))
        for m, style in [("fixed_filter", "o-r"), ("ccbf", "s-b")]:
            ax.plot(CONFIG.T_GRID, curve[m], style, label=m)
        ax.set_xscale("log"); ax.set_xlabel("sequence length T"); ax.set_ylabel("ECE (final step)")
        ax.set_title("ECE vs sequence length  (Theorem 1: fixed grows, CCBF flat)"); ax.legend()
        out = os.path.join(CONFIG.WORKING_DIR, "A3_ece_vs_T.png"); fig.tight_layout(); fig.savefig(out, dpi=110)
        print("saved", out); plt.show()
    except Exception as e:
        print("(plot skipped:", repr(e)[:60], ")")
    return curve

if RUN_A:
    RESULTS2["A3_ece_vs_T"] = exp_A3_ece_vs_T(REF, QUERIES, COND)
    log("A.3", "ok", "ECE-vs-T computed (see plot)")

In [ ]:
# === A.4 — Open-set / off-map AUROC ===================================================
def exp_A4_offmap(ref, queries, COND, frac=None):
    frac = frac or CONFIG.OFFMAP_MAP_FRACTION
    rows = {}
    agg = {m: {"score": [], "label": []} for m in ["ccbf_null", "fixed_null", "sue"]}
    for qs in queries:
        full = DESC[qs]; M = DESC[ref].shape[0]; Mmap = int(M * frac)
        ref_map = DESC[ref][:Mmap]
        sim = cosine_sim(full, ref_map)                       # (T, Mmap)
        label = (np.arange(len(full)) >= Mmap).astype(int)    # off-map if true place beyond the map
        _, _, nc = run_ccbf(sim, full, COND)
        _, _, nf = run_place_filter(sim, fit_global_tau(ref, queries))
        sue = sue_uncertainty(sim)
        for m, sc in [("ccbf_null", nc), ("fixed_null", nf), ("sue", sue)]:
            agg[m]["score"].append(sc); agg[m]["label"].append(label)
        rows[qs] = {"ccbf_null": auroc(nc, label), "fixed_null": auroc(nf, label), "sue": auroc(sue, label)}
    overall = {}
    for m in agg:
        s = np.concatenate(agg[m]["score"]); l = np.concatenate(agg[m]["label"])
        overall[m] = {"AUROC": auroc(s, l), "AUPR": average_precision(s, l)}
    print("off-map detection (map = first {:.0%} of route):".format(frac))
    for m, v in overall.items(): print(f"  {m:<12} AUROC={v['AUROC']:.3f}  AUPR={v['AUPR']:.3f}")
    return {"overall": overall, "per_season": rows}

if RUN_A:
    RESULTS2["A4_offmap"] = exp_A4_offmap(REF, QUERIES, COND)
    log("A.4", "ok", f"CCBF off-map AUROC={RESULTS2['A4_offmap']['overall']['ccbf_null']['AUROC']:.3f}")

In [ ]:
# === A.5 — Label-free test-time domain adaptation ====================================
def exp_A5_tta(ref, queries, COND):
    names = COND["names"]
    # (a) per-condition belief accuracy (query is a single condition -> belief should pick it)
    accs = {}
    for qs in queries:
        sim, gt, _, _ = episode(ref, qs)
        _, _, _, traj = run_ccbf(sim, DESC[qs], COND, return_traj=True)
        if qs in names:
            accs[qs] = float((traj.argmax(1) == names.index(qs)).mean())
    print("condition-belief accuracy (label-free):", {k: round(v, 3) for k, v in accs.items()})
    # (b) condition-drift tracking: concat two conditions, expect a sharp belief switch
    drift = None
    if len(queries) >= 1 and ref in names:
        A, B = ref, queries[0]
        if A in names and B in names and A in DESC and B in DESC:
            dA, dB = DESC[A], DESC[B]; M = dA.shape[0]; H = M // 2
            qd = np.concatenate([dA[:H], dB[H:]], 0)
            true_c = np.array([names.index(A)] * H + [names.index(B)] * (M - H))
            sim = cosine_sim(qd, DESC[A])
            _, _, _, traj = run_ccbf(sim, qd, COND, return_traj=True)
            drift = float((traj.argmax(1) == true_c).mean())
            print(f"drift tracking acc = {drift:.3f} (chance={1/len(names):.3f})")
    return {"belief_acc": accs, "drift_acc": drift}

if RUN_A:
    RESULTS2["A5_tta"] = exp_A5_tta(REF, QUERIES, COND)
    log("A.5", "ok", f"drift_acc={RESULTS2['A5_tta']['drift_acc']}")

## Experiment B / C — RobotCar Seasons v2 & Extended CMU (best-effort, self-probing)

These are **pose** benchmarks whose GT schema we verify at runtime rather than assume. Each section first **probes**
the GT file (prints a few lines so the loader can be finalized), then runs a guarded single-image hard-condition
evaluation (recall + ECE + SUE). The full **sequence/CCBF** evaluation with pose-indexed maps is finalized in NB3
once the probe confirms the format — this keeps NB2 honest about what is verified.

In [ ]:
# === B — RobotCar Seasons v2 (hard conditions: night / rain / snow / ...) ==============
def _isfloat(x):
    try: float(x); return True
    except Exception: return False

def probe_and_run_robotcar():
    root = PATHS["robotcar"]
    if not root:
        log("B.robotcar", "warn", "RobotCar not attached; skipped"); return None
    txt = None
    for name in ["robotcar_v2_train.txt", "robotcar_v2_test.txt"]:
        c = os.path.join(root, name)
        if os.path.isfile(c): txt = c; break
    if not txt:
        hits = glob.glob(os.path.join(root, "**", "*train*.txt"), recursive=True)
        txt = hits[0] if hits else None
    if not txt:
        log("B.robotcar", "warn", "no train txt found; skipped"); return None
    # --- PROBE: print the format so the loader can be confirmed ---
    print("=== RobotCar GT probe:", txt, "===")
    with open(txt) as f: head = [next(f, "").rstrip() for _ in range(5)]
    for h in head: print("   ", h[:160])
    # --- parse (path + pose); xyz = translation ---
    entries = []
    with open(txt) as f:
        for line in f:
            tok = line.split()
            if len(tok) < 4: continue
            nums = [float(x) for x in tok[1:] if _isfloat(x)]
            if len(nums) >= 7: xyz = np.array(nums[4:7])
            elif len(nums) >= 3: xyz = np.array(nums[-3:])
            else: continue
            entries.append((tok[0], xyz))
    print(f"parsed {len(entries)} pose entries")
    if not entries:
        log("B.robotcar", "warn", "could not parse poses; need format fix (see probe)"); return None

    def cond_of(p):
        c = p.replace("\\", "/").split("/")[0].lower()
        return c
    def resolve(p):
        p = p.replace("\\", "/")
        cands = [os.path.join(root, "images", p)]
        parts = p.split("/")
        if len(parts) >= 2:
            cands.append(os.path.join(root, "images", parts[0], parts[0], *parts[1:]))
        for c in cands:
            if os.path.isfile(c): return c
        return None

    by_cond = defaultdict(list)
    for p, xyz in entries:
        rp = resolve(p)
        if rp: by_cond[cond_of(p)].append((rp, xyz))
    print("resolved images per condition:", {k: len(v) for k, v in by_cond.items()})
    ref_key = next((k for k in by_cond if CONFIG.ROBOTCAR_REF_COND in k), None)
    if ref_key is None or len(by_cond[ref_key]) < 10:
        log("B.robotcar", "warn", f"reference cond '{CONFIG.ROBOTCAR_REF_COND}' not resolvable; probe only");
        return {"probe_only": True, "counts": {k: len(v) for k, v in by_cond.items()}}

    cap = 300 if CONFIG.SMOKE_TEST else 4000
    def take(lst):
        idx = np.linspace(0, len(lst) - 1, min(cap, len(lst))).round().astype(int)
        return [lst[i] for i in sorted(set(idx.tolist()))]
    ref = take(by_cond[ref_key]); ref_xyz = np.stack([x for _, x in ref])
    ref_desc = extract_paths([p for p, _ in ref], list(range(len(ref))), f"rc_ref_{'s' if CONFIG.SMOKE_TEST else 'f'}")
    out = {}
    for qc in CONFIG.ROBOTCAR_QUERY_CONDS:
        key = next((k for k in by_cond if qc in k and k != ref_key), None)
        if key is None or len(by_cond[key]) < 10: continue
        q = take(by_cond[key]); q_xyz = np.stack([x for _, x in q])
        q_desc = extract_paths([p for p, _ in q], list(range(len(q))), f"rc_{qc}_{'s' if CONFIG.SMOKE_TEST else 'f'}")
        sim = cosine_sim(q_desc, ref_desc)
        d = np.sqrt(((q_xyz[:, None, :] - ref_xyz[None, :, :]) ** 2).sum(-1))   # (Nq, Nref)
        correct_set = d <= CONFIG.POSE_TOL_M
        top1 = sim.argmax(1)
        r1 = float(correct_set[np.arange(len(q)), top1].mean())
        # ECE: calibrated-ish confidence via a quick temperature on this condition
        gt_near = d.argmin(1)
        tau, _ = fit_temperature(sim, np.where(correct_set.any(1), gt_near, -1))
        pr = np.exp(sim / tau - _logsumexp(sim / tau, axis=1)[:, None])
        corr = correct_set[np.arange(len(q)), pr.argmax(1)].astype(float)
        ece, _ = ece_from_conf(pr.max(1), corr, CONFIG.ECE_BINS)
        out[qc] = {"R@1": r1, "ECE": ece, "n": len(q)}
        print(f"  RobotCar {qc:<12} R@1={r1:.3f}  ECE={ece:.3f}  (n={len(q)})")
    log("B.robotcar", "ok" if out else "warn", f"conditions evaluated: {list(out)}")
    return {"ref": ref_key, "per_condition": out}

RESULTS2 = globals().get("RESULTS2", OrderedDict())
try:
    RESULTS2["B_robotcar"] = probe_and_run_robotcar()
except Exception as e:
    import traceback; traceback.print_exc(); log("B.robotcar", "warn", f"error: {repr(e)[:100]}")

In [ ]:
# === C — Extended CMU Seasons (probe only in NB2; full eval in NB3) ====================
def probe_cmu():
    root = PATHS["cmu"]
    if not root:
        log("C.cmu", "warn", "CMU not attached; skipped"); return None
    slices = sorted(glob.glob(os.path.join(root, "slice*")))[:3]
    info = {}
    for sl in slices:
        base = sl
        inner = os.path.join(sl, os.path.basename(sl))
        if os.path.isdir(inner): base = inner
        pose_dir = os.path.join(base, "camera-poses")
        db = os.path.join(base, "database"); q = os.path.join(base, "query")
        pose_files = glob.glob(os.path.join(pose_dir, "*")) if os.path.isdir(pose_dir) else []
        n_db = len(_imgs(db)) if os.path.isdir(db) else 0
        n_q = len(_imgs(q)) if os.path.isdir(q) else 0
        head = []
        if pose_files:
            try:
                with open(pose_files[0]) as f: head = [next(f, "").rstrip()[:140] for _ in range(3)]
            except Exception: pass
        info[os.path.basename(sl)] = {"pose_files": len(pose_files), "db_imgs": n_db, "query_imgs": n_q, "pose_head": head}
        print(f"CMU {os.path.basename(sl)}: db={n_db} query={n_q} pose_files={len(pose_files)}")
        for h in head: print("     pose:", h)
    log("C.cmu", "ok", f"probed {len(info)} slices (full CCBF eval deferred to NB3)")
    return info

try:
    RESULTS2["C_cmu_probe"] = probe_cmu()
except Exception as e:
    log("C.cmu", "warn", f"error: {repr(e)[:100]}")

## Summary & health gate

In [ ]:
# === Summary ===========================================================================
def _save_results():
    try:
        import pickle
        path = os.path.join(CONFIG.ckpt_dir, "nb2_results.pkl")
        pickle.dump({"results": RESULTS2, "report": REPORT, "extractor": EXTRACTOR_INFO,
                     "config": asdict(CONFIG)}, open(path, "wb"))
        print("saved results ->", path)
    except Exception as e:
        print("(save skipped:", repr(e)[:60], ")")

print("=" * 66); print("NB2 HEADLINE RESULTS"); print("=" * 66)
if "A1_comparison" in RESULTS2:
    print("\n[A1] Multi-condition localization (Nordland):")
    print(f"  {'method':<16}{'R@1':>8}{'ECE':>8}")
    for m, v in RESULTS2["A1_comparison"].items():
        print(f"  {m:<16}{v['R@1']:>8.3f}{v['ECE']:>8.3f}")
    a = RESULTS2["A1_comparison"]
    if "fixed_filter" in a and "ccbf" in a:
        print(f"  -> ECE improvement (fixed - ccbf): {a['fixed_filter']['ECE']-a['ccbf']['ECE']:+.3f}")
if "A4_offmap" in RESULTS2 and RESULTS2["A4_offmap"]:
    print("\n[A4] Open-set / off-map AUROC:")
    for m, v in RESULTS2["A4_offmap"]["overall"].items():
        print(f"  {m:<12} AUROC={v['AUROC']:.3f}  AUPR={v['AUPR']:.3f}")
if "A5_tta" in RESULTS2:
    print("\n[A5] Label-free TTA: drift_acc =", RESULTS2["A5_tta"].get("drift_acc"))
if "B_robotcar" in RESULTS2 and RESULTS2["B_robotcar"] and RESULTS2["B_robotcar"].get("per_condition"):
    print("\n[B] RobotCar hard-condition single-image recall:")
    for c, v in RESULTS2["B_robotcar"]["per_condition"].items():
        print(f"  {c:<12} R@1={v['R@1']:.3f} ECE={v['ECE']:.3f}")

n_ok = sum(1 for v in REPORT.values() if v["status"] == "ok")
n_warn = sum(1 for v in REPORT.values() if v["status"] == "warn")
n_fail = sum(1 for v in REPORT.values() if v["status"] == "fail")
print("\n" + "=" * 66)
print(f"REPORT: {n_ok} ok / {n_warn} warn / {n_fail} fail  | extractor={EXTRACTOR_INFO['name']}")
for s, v in REPORT.items():
    if v["status"] in ("warn", "fail"): print(f"  [{v['status']}] {s}: {v['msg']}")
_save_results()
print("""
Next steps:
  - If Experiment A shows CCBF ECE < fixed_filter ECE and a rising ECE-vs-T curve for the fixed filter,
    that is the empirical signature of Theorem 1 -> the paper's core claim is supported.
  - Paste the [B] RobotCar probe lines + [C] CMU pose-head lines so the pose loaders can be finalized (NB3),
    adding real hard-condition sequence/pose results.
  - For full scale: set CONFIG.SMOKE_TEST=False (caches all 4 Nordland seasons; the per-step Python filter
    will be vectorized to GPU in NB3 for the 143k-frame run).
""")

### What NB2 establishes
- The **four headline experiments** on clean, multi-condition Nordland: calibration (ECE + reliability),
  **Theorem 1** signature (ECE-vs-T), **open-set AUROC**, and **label-free TTA**.
- A **self-probing** entry point for RobotCar / CMU hard conditions, honest about what is verified vs. deferred.

### Report back for NB3
Paste the **[B] RobotCar probe** lines, the **[C] CMU pose-head** lines, and the **A1/A3/A4** numbers (or the
executed `.ipynb`). That finalizes the pose loaders and tells us whether the calibration story holds on real
hard conditions — the last piece before writing the experiments section.